In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pickle
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import gc
import os


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BASE_PATH = '/content/drive/MyDrive/embeddings_assignment/processed_data/'
OUTPUT_PATH = '/content/drive/MyDrive/embeddings_assignment/FastText_pipeline_results/'


In [ ]:
class NERDataset(Dataset):
    def __init__(self, data, vocab=None):
        self.sentences = data['tokens']; self.tags = data['tags']
        self.vocab = {'<PAD>': 0, '<UNK>': 1} if vocab is None else vocab
        if vocab is None:
            for s in self.sentences:
                for t in s:
                    if t not in self.vocab: self.vocab[t] = len(self.vocab)
    def __len__(self): return len(self.sentences)
    def __getitem__(self, idx):
        return torch.tensor([self.vocab.get(t, self.vocab['<UNK>']) for t in self.sentences[idx]]), torch.tensor(self.tags[idx])

In [ ]:
def collate_fn(batch):
    s, t = zip(*batch)
    return pad_sequence(s, batch_first=True, padding_value=0), pad_sequence(t, batch_first=True, padding_value=-100)

In [ ]:
class FastTextBiLSTM(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, target_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim, hidden_dim // 2, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hidden_dim, target_size)
    def forward(self, x): return self.fc(self.lstm(self.embedding(x))[0])

In [ ]:
tokenizers = ['whitespace', 'nltk', 'wordpiece']
TAG_LABELS = ['O', 'B-Disease', 'I-Disease']

In [9]:
os.makedirs(OUTPUT_PATH, exist_ok=True)
print(f"Output directory '{OUTPUT_PATH}' ensured to exist.")

for tok in tokenizers:
    print(f"\n================ RUNNING: FASTTEXT + {tok.upper()} ================")
    with open(f'{BASE_PATH}train_{tok}.pkl', 'rb') as f: train_pkl = pickle.load(f)
    with open(f'{BASE_PATH}val_{tok}.pkl', 'rb') as f: val_pkl = pickle.load(f)
    with open(f'{BASE_PATH}test_{tok}.pkl', 'rb') as f: test_pkl = pickle.load(f)

    train_dataset = NERDataset(train_pkl)
    val_dataset = NERDataset(val_pkl, vocab=train_dataset.vocab)
    test_dataset = NERDataset(test_pkl, vocab=train_dataset.vocab)

    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

    # Calculate Weights
    flat_tags = [t for tags in train_pkl['tags'] for t in tags if t != -100]
    class_counts = np.bincount(flat_tags, minlength=len(TAG_LABELS))
    class_counts = np.where(class_counts == 0, 1, class_counts)
    class_weights = sum(class_counts) / (len(TAG_LABELS) * class_counts)
    class_weights_tensor = torch.FloatTensor(class_weights).to(device)

    model = FastTextBiLSTM(len(train_dataset.vocab), 300, 128, len(TAG_LABELS)).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor, ignore_index=-100)

    t_loss, v_loss, t_acc, v_acc = [], [], [], []
    for epoch in range(8):
        model.train()
        loss_val, ok, tot = 0, 0, 0
        for seqs, tags in train_loader:
            seqs, tags = seqs.to(device), tags.to(device)
            optimizer.zero_grad()
            out = model(seqs)
            loss = criterion(out.view(-1, out.shape[-1]), tags.view(-1))
            loss.backward()
            optimizer.step()
            loss_val += loss.item()
            preds = torch.argmax(out, dim=-1)
            mask = tags != -100
            ok += (preds[mask] == tags[mask]).sum().item()
            tot += mask.sum().item()

        model.eval()
        v_loss_val, v_ok, v_tot = 0, 0, 0
        with torch.no_grad():
            for seqs, tags in val_loader:
                seqs, tags = seqs.to(device), tags.to(device)
                out = model(seqs)
                loss = criterion(out.view(-1, out.shape[-1]), tags.view(-1))
                v_loss_val += loss.item()
                preds = torch.argmax(out, dim=-1)
                mask = tags != -100
                v_ok += (preds[mask] == tags[mask]).sum().item()
                v_tot += mask.sum().item()

        t_loss.append(loss_val/len(train_loader)); t_acc.append(ok/max(1,tot))
        v_loss.append(v_loss_val/len(val_loader)); v_acc.append(v_ok/max(1,v_tot))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(t_loss, 'c-', label='Train'); ax1.plot(v_loss, 'r-', label='Val'); ax1.set_title('Loss'); ax1.legend()
    ax2.plot(t_acc, 'c-', label='Train'); ax2.plot(v_acc, 'r-', label='Val'); ax2.set_title('Accuracy'); ax2.legend()
    plt.savefig(f'{OUTPUT_PATH}fasttext_{tok}_loss_accuracy.png')
    plt.close(fig)

    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for seqs, tags in test_loader:
            seqs = seqs.to(device)
            preds = torch.argmax(model(seqs), dim=-1).cpu()
            mask = tags != -100
            y_true.extend(tags[mask].tolist()); y_pred.extend(preds[mask].tolist())

    report = classification_report(y_true, y_pred, target_names=TAG_LABELS, labels=range(len(TAG_LABELS)), zero_division=0)
    print(report)
    with open(f'{OUTPUT_PATH}fasttext_{tok}_classification_report.txt', 'w') as f:
        f.write(report)

    sns.heatmap(confusion_matrix(y_true, y_pred, labels=range(len(TAG_LABELS))), annot=True, fmt='d', cmap='Oranges', xticklabels=TAG_LABELS, yticklabels=TAG_LABELS)
    plt.savefig(f'{OUTPUT_PATH}fasttext_{tok}_confusion_matrix.png')
    plt.close(plt.gcf())

    del model, train_loader, val_loader, test_loader
    gc.collect(); torch.cuda.empty_cache()

Output directory '/content/drive/MyDrive/embeddings_assignment/FastText_pipeline_results/' ensured to exist.

================ RUNNING: FASTTEXT + WHITESPACE ================
              precision    recall  f1-score   support

           O       0.99      0.98      0.98    117589
   B-Disease       0.68      0.72      0.70      4424
   I-Disease       0.64      0.63      0.64      2737

    accuracy                           0.97    124750
   macro avg       0.77      0.78      0.77    124750
weighted avg       0.97      0.97      0.97    124750


================ RUNNING: FASTTEXT + NLTK ================
              precision    recall  f1-score   support

           O       0.98      0.99      0.99    117595
   B-Disease       0.76      0.67      0.71      4424
   I-Disease       0.72      0.61      0.66      2737

    accuracy                           0.97    124756
   macro avg       0.82      0.76      0.79    124756
weighted avg       0.97      0.97      0.97    124756


==